In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import pickle

# Clean data load karo
df = pd.read_csv('data/disasters_clean.csv')

print("Data loaded!")
print("Shape:", df.shape)
df.head()

Data loaded!
Shape: (16126, 13)


,Year,Disaster Group,Disaster Subgroup,Disaster Type,Country,Continent,Region,Total Deaths,No Injured,No Affected,No Homeless,Total Affected,Total Damages ('000 US$)
0,1900,Natural,Climatological,Drought,Cabo Verde,Africa,Western Africa,11000.0,0.0,0.0,0.0,0.0,0.0
1,1900,Natural,Climatological,Drought,India,Asia,Southern Asia,1250000.0,0.0,0.0,0.0,0.0,0.0
2,1902,Natural,Geophysical,Earthquake,Guatemala,Americas,Central America,2000.0,0.0,0.0,0.0,0.0,25000.0
3,1902,Natural,Geophysical,Volcanic activity,Guatemala,Americas,Central America,1000.0,0.0,0.0,0.0,0.0,0.0
4,1902,Natural,Geophysical,Volcanic activity,Guatemala,Americas,Central America,6000.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Severity score banao
df['severity_score'] = (
    np.log1p(df['Total Deaths']) +
    np.log1p(df['Total Affected'])
)

# Severity ko 3 classes mein divide karo
# Low = 0, Medium = 1, High = 2
df['severity_class'] = pd.cut(
    df['severity_score'],
    bins=3,
    labels=[0, 1, 2]
).astype(int)

print("Severity Classes:")
print(df['severity_class'].value_counts())

# Disaster Type encode karo (text to number)
le = LabelEncoder()
df['Disaster_Type_Encoded'] = le.fit_transform(df['Disaster Type'])

print("\nDisaster Types Encoded:")
print(df[['Disaster Type', 'Disaster_Type_Encoded']].drop_duplicates())

Severity Classes:
severity_class
0    10665
1     5311
2      150
Name: count, dtype: int64

Disaster Types Encoded:
               Disaster Type  Disaster_Type_Encoded
0                    Drought                      1
2                 Earthquake                      2
3          Volcanic activity                     13
5        Mass movement (dry)                     11
7                      Storm                     12
12                     Flood                      5
16                  Epidemic                      3
25                 Landslide                     10
33                  Wildfire                     14
129     Extreme temperature                       4
209                      Fog                      6
890       Insect infestation                      9
12856                 Impact                      8
13456        Animal accident                      0
15814  Glacial lake outburst                      7


In [ ]:
# Features aur Target define karo
X = df[['Year', 'Total Deaths', 'Total Affected', 
        "Total Damages ('000 US$)", 'Disaster_Type_Encoded']]
y = df['severity_class']

# Train/Test split karo
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training data size:", X_train.shape)
print("Testing data size:", X_test.shape)

# XGBoost Model banao aur train karo
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)
print("\n✅ Model training complete!")

# Accuracy check karo
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"🎯 Accuracy: {accuracy*100:.2f}%")

Training data size: (12900, 5)
Testing data size: (3226, 5)

✅ Model training complete!
🎯 Accuracy: 99.47%


In [ ]:
# Model save karo
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("✅ model.pkl saved!")

# Label Encoder bhi save karo
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print("✅ label_encoder.pkl saved!")

✅ model.pkl saved!
✅ label_encoder.pkl saved!


In [2]:
import pickle
import numpy as np

# Model aur Label Encoder load karo
with open('model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

print("✅ Model loaded!")
print("✅ Label Encoder loaded!")
print("Disaster Types:", le.classes_)

✅ Model loaded!
✅ Label Encoder loaded!
Disaster Types: ['Animal accident' 'Drought' 'Earthquake' 'Epidemic'
 'Extreme temperature ' 'Flood' 'Fog' 'Glacial lake outburst' 'Impact'
 'Insect infestation' 'Landslide' 'Mass movement (dry)' 'Storm'
 'Volcanic activity' 'Wildfire']


In [3]:
# Prediction Function
def predict_disaster(disaster_type, year, deaths, affected, damages):
    print(f"=== Disaster Prediction ===")
    print(f"Disaster Type: {disaster_type}")
    print(f"Year: {year}")
    print(f"Deaths: {deaths}")
    print(f"Affected: {affected}")
    print(f"Damages: ${damages},000 USD")
    print()
    
    # Encode disaster type
    disaster_encoded = le.transform([disaster_type])[0]
    
    # Features
    X_new = [[year, deaths, affected, damages, disaster_encoded]]
    
    # Predict
    pred = model.predict(X_new)[0]
    pred_proba = model.predict_proba(X_new)[0]
    
    labels = {0: "Low 🟢", 1: "Medium 🟡", 2: "High 🔴"}
    
    print(f"Predicted Severity: {labels[pred]}")
    print(f"\nProbability:")
    print(f"  Low:    {pred_proba[0]*100:.2f}%")
    print(f"  Medium: {pred_proba[1]*100:.2f}%")
    print(f"  High:   {pred_proba[2]*100:.2f}%")

# Example Predictions
print("PREDICTION 1:")
predict_disaster("Flood", 2024, 500, 100000, 5000000)
print("\n" + "="*40 + "\n")

print("PREDICTION 2:")
predict_disaster("Earthquake", 2023, 10000, 500000, 10000000)
print("\n" + "="*40 + "\n")

print("PREDICTION 3:")
predict_disaster("Drought", 2022, 100, 50000, 1000000)

PREDICTION 1:
=== Disaster Prediction ===
Disaster Type: Flood
Year: 2024
Deaths: 500
Affected: 100000
Damages: $5000000,000 USD

Predicted Severity: Medium 🟡

Probability:
  Low:    0.00%
  Medium: 100.00%
  High:   0.00%


PREDICTION 2:
=== Disaster Prediction ===
Disaster Type: Earthquake
Year: 2023
Deaths: 10000
Affected: 500000
Damages: $10000000,000 USD

Predicted Severity: High 🔴

Probability:
  Low:    0.08%
  Medium: 15.46%
  High:   84.46%


PREDICTION 3:
=== Disaster Prediction ===
Disaster Type: Drought
Year: 2022
Deaths: 100
Affected: 50000
Damages: $1000000,000 USD

Predicted Severity: Medium 🟡

Probability:
  Low:    0.00%
  Medium: 100.00%
  High:   0.00%
